# Predictive Analysis- Feature Engineering (REGRESSION)

In this notebook, we extract additional features from our RUL regression dataset by transforming and creating meaningful input variables. Then, we split our dataset into test/train/val datasets for regression model training and evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json


### Data Loading

Loading the regression master dataset with RUL targets


In [ ]:
# loading master regression dataset
file_path = '/content/drive/MyDrive/predictive-analysis-data/regression/master_rul_24h.csv'

df = pd.read_csv(file_path)
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"Data loaded with {len(df):,} rows and {df.shape[1]} columns.")
print(f"\nRUL Statistics:")
print(df['RUL_hours'].describe())


Data loaded with 876,100 rows and 25 columns.

RUL Statistics:
count    17902.000000
mean        11.970506
std          7.198280
min          0.000000
25%          6.000000
50%         12.000000
75%         18.000000
max         24.000000
Name: RUL_hours, dtype: float64


### Creating the Features


#### Rolling Window Feature

We are capturing each machine's sensor behavior patterns over time by computing the mean and standard deviation of the sensors over the past 6 and 24 hours.<br>
This enables our model to understand how a machine has been behaving recently and see if its sensors are in a stable stage or not.


In [ ]:
# sort df by machine id then datetime
df = df.sort_values(['machineID', 'datetime']).reset_index(drop=True)

sensors = ['volt', 'rotate', 'pressure', 'vibration']
windows = [6, 24]
stats = ['mean', 'std']

rolling_features = []

for sensor in sensors:
    for window in windows:
        # grouping by machine and calculating the rolling stats
        roll = df.groupby('machineID')[sensor].rolling(
            window=window,
            min_periods=1
        )

        for stat in stats:
            # eg: volt_rolling_24h_stat
            feature_name = f'{sensor}_rolling_{window}h_{stat}'

            if stat == 'mean':
                df[feature_name] = roll.mean().reset_index(0, drop=True)
            elif stat == 'std':
                df[feature_name] = roll.std().reset_index(0, drop=True)

            rolling_features.append(feature_name)

# handle NaN values that gets created on the first std calculation of every window
std_columns = [col for col in rolling_features if '_std' in col]
df[std_columns] = df[std_columns].fillna(0)

print(f"Total {len(rolling_features)} rolling window features created")


Total 16 rolling window features created


#### Lag Features

Here, we are creating lag features to capture each machine's sensor values from the past few hours, along with the rate of change in those values over the past hour. These features help our model track trends and sudden shifts in sensor behavior over time.


In [ ]:
lag_features = []
lag_hours = [1, 3]

for sensor in sensors:
    # get the sensor's value from 'l' hours before
    for lag in lag_hours:
        feature_name = f'{sensor}_lag_{lag}h'
        df[feature_name] = df.groupby('machineID')[sensor].shift(lag)
        lag_features.append(feature_name)

    # get the rate of change in sensor value from previous (1) hour
    feature_name = f'{sensor}_change_1h'
    df[feature_name] = df[sensor] - df.groupby('machineID')[sensor].shift(1)
    lag_features.append(feature_name)

# handle any NaN values at start of each machine's time series
df[lag_features] = df[lag_features].fillna(0)

print(f"Created {len(lag_features)} lag features")


Created 12 lag features


#### Critical Events Feature

We capture the critical events of each machine over time. Specifically, for each machine, we compute the:
- Number of errors in the last 24 hours
- Hours since last error and last maintenance
- Days since last failure
- Total number of errors, maintenance and failures encountered to date


In [ ]:
event_features = []

# Error count in last 24 hours
df['error_count_last_24h'] = df.groupby('machineID')['has_error'].rolling(
    window=24,
    min_periods=1
).sum().reset_index(0, drop=True)
event_features.append('error_count_last_24h')

# Hours since last error
df['hours_since_last_error'] = np.nan
for machine_id in df['machineID'].unique():
    mask = df['machineID'] == machine_id
    machine_df = df[mask].copy()
    error_indices = machine_df[machine_df['has_error'] == 1].index

    for i in machine_df.index:
        prev_errors = error_indices[error_indices < i]
        if len(prev_errors) > 0:
            last_error_index = prev_errors[-1]
            last_error_time = machine_df.loc[last_error_index, 'datetime']
            current_time = machine_df.loc[i, 'datetime']
            dif_hours = (current_time - last_error_time).total_seconds()/3600
            df.loc[i, 'hours_since_last_error'] = dif_hours
        else:
            df.loc[i, 'hours_since_last_error'] = 999

event_features.append('hours_since_last_error')


In [ ]:
# Hours since last maintenance
df['hours_since_last_maintenance'] = np.nan
for machine_id in df['machineID'].unique():
    mask = df['machineID'] == machine_id
    machine_df = df[mask].copy()
    maint_indices = machine_df[machine_df['has_maintenance'] == 1].index

    for i in machine_df.index:
        prev_maint = maint_indices[maint_indices < i]
        if len(prev_maint) > 0:
            last_maint_index = prev_maint[-1]
            last_maint_time = machine_df.loc[last_maint_index, 'datetime']
            current_time = machine_df.loc[i, 'datetime']
            dif_hours = (current_time - last_maint_time).total_seconds()/3600
            df.loc[i, 'hours_since_last_maintenance'] = dif_hours
        else:
            df.loc[i, 'hours_since_last_maintenance'] = 999

event_features.append('hours_since_last_maintenance')


In [ ]:
# Days since last failure
df['days_since_last_failure'] = np.nan
for machine_id in df['machineID'].unique():
    mask = df['machineID'] == machine_id
    machine_df = df[mask].copy()
    failure_indices = machine_df[machine_df['has_failure'] == 1].index

    for i in machine_df.index:
        prev_failures = failure_indices[failure_indices < i]
        if len(prev_failures) > 0:
            last_failure_index = prev_failures[-1]
            last_failure_time = machine_df.loc[last_failure_index, 'datetime']
            current_time = machine_df.loc[i, 'datetime']
            dif_days = (current_time - last_failure_time).total_seconds()/(3600*24)
            df.loc[i, 'days_since_last_failure'] = dif_days
        else:
            df.loc[i, 'days_since_last_failure'] = 999

event_features.append('days_since_last_failure')

# Cumulative event counts
df['total_errors_to_date'] = df.groupby('machineID')['has_error'].cumsum()
df['total_maintenances_to_date'] = df.groupby('machineID')['has_maintenance'].cumsum()
df['total_failures_to_date'] = df.groupby('machineID')['has_failure'].cumsum()

event_features.extend(['total_errors_to_date', 'total_maintenances_to_date', 'total_failures_to_date'])

print(f"Total {len(event_features)} event history features created")


Total 7 event history features created


### Machine-specific Features

Here, we capture & convert some important machine-specific features for our ML model, like:
- One-hot encode the 4 different models of machine
- Get age squared as a polynomial feature to capture the non-linear relation between age & RUL
- Machine's sensor baseline deviation: how does the current sensor reading compare to its average


In [ ]:
machine_features = []

# one-hot encoding the 4 machine models
dummies_model = pd.get_dummies(df['model'], drop_first=True)
df = pd.concat([df, dummies_model], axis=1)
machine_features.extend(dummies_model.columns.tolist())

# compute age squared
df['age_squared'] = df['age'] ** 2
machine_features.append('age_squared')

# compute machine baseline deviations
for sensor in sensors:
    machine_avg = df.groupby('machineID')[sensor].transform('mean')
    feature_name = f'{sensor}_deviation_from_machine_avg'
    df[feature_name] = df[sensor] - machine_avg
    machine_features.append(feature_name)

print(f"Total {len(machine_features)} machine-specific features created.")

# Summary
all_new_features = (rolling_features + lag_features + event_features + machine_features)
print(f"\nTotal new features created: {len(all_new_features)}")
print(f"Rolling window: {len(rolling_features)}")
print(f"Lag features: {len(lag_features)}")
print(f"Event history: {len(event_features)}")
print(f"Machine-specific: {len(machine_features)}")


Total 8 machine-specific features created.

Total new features created: 43
Rolling window: 16
Lag features: 12
Event history: 7
Machine-specific: 8


### Train/Val/Test split (Temporal)

Here, we perform a temporal split on our master dataset as we have time series data, so we cannot do a random split. <br>
We choose the split as:
- Train: 2015-01-01 to 2015-10-31 (80%)
- Val: 2015-11-01 to 2015-11-30 (10%)
- Test: 2015-12-01 to 2015-12-31 (10%)


In [ ]:
# defining split dates
train_end = pd.Timestamp('2015-10-31 23:59:59')
val_end = pd.Timestamp('2015-11-30 23:59:59')

# performing the splits
train_mask = df['datetime'] <= train_end
val_mask = (df['datetime'] > train_end) & (df['datetime'] <= val_end)
test_mask = df['datetime'] > val_end

train_df = df[train_mask].copy()
val_df = df[val_mask].copy()
test_df = df[test_mask].copy()

print(f"\nSplit Summary:")
print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

# RUL distribution across splits
print(f"\nRUL Distribution:")
print(f"Train - Mean: {train_df['RUL_hours'].mean():.2f}, Median: {train_df['RUL_hours'].median():.2f}, Std: {train_df['RUL_hours'].std():.2f}")
print(f"Val   - Mean: {val_df['RUL_hours'].mean():.2f}, Median: {val_df['RUL_hours'].median():.2f}, Std: {val_df['RUL_hours'].std():.2f}")
print(f"Test  - Mean: {test_df['RUL_hours'].mean():.2f}, Median: {test_df['RUL_hours'].median():.2f}, Std: {test_df['RUL_hours'].std():.2f}")



Split Summary:
Train: 729,000 rows
Val:   72,000 rows
Test:  75,100 rows

RUL Distribution:
Train - Mean: 11.98, Median: 12.00, Std: 7.19
Val   - Mean: 11.96, Median: 12.00, Std: 7.22
Test  - Mean: 11.91, Median: 12.00, Std: 7.24


In [ ]:
# Define feature columns (exclude identifiers, timestamps, failure flags, and RUL targets)
exclude_cols = ['machineID', 'datetime', 'model',
                'failure_comp1', 'failure_comp2', 'failure_comp3', 'failure_comp4',
                'has_error',
                'RUL_hours']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Total {len(feature_cols)} features.")

# Separate features & target
X_train = train_df[feature_cols]
y_train = train_df['RUL_hours']

X_val = val_df[feature_cols]
y_val = val_df['RUL_hours']

X_test = test_df[feature_cols]
y_test = test_df['RUL_hours']

# Store metadata for later
train_meta = train_df[['machineID', 'datetime', 'RUL_hours']].copy()
val_meta = val_df[['machineID', 'datetime', 'RUL_hours']].copy()
test_meta = test_df[['machineID', 'datetime', 'RUL_hours']].copy()


Total 59 features.


### Feature Scaling

We are scaling our dataset on the training data to standardize the range of all feature variables.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit on training data
scaler.fit(X_train)

# transform all datasets
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# converting to df for easier export
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=feature_cols, index=X_val.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"Scaling Verification:")
print(f"Train set- Mean: {X_train_scaled_df.mean().mean():.4f}, Std: {X_train_scaled_df.std().mean():.4f}")
print(f"Val set - Mean: {X_val_scaled_df.mean().mean():.4f}, Std: {X_val_scaled_df.std().mean():.4f}")
print(f"Test set - Mean: {X_test_scaled_df.mean().mean():.4f}, Std: {X_test_scaled_df.std().mean():.4f}")


Scaling Verification:
Train set- Mean: -0.0000, Std: 1.0000
Val set - Mean: 0.0727, Std: 0.9636
Test set - Mean: 0.0920, Std: 0.9731


In [ ]:
import pickle

with open('/content/drive/MyDrive/predictive-analysis-data/regression/regression_scaler.pkl', 'wb') as f:
  pickle.dump(scaler, f)

print("\n Scaler saved successfully!")


 Scaler saved successfully!


### Exporting Datasets


In [ ]:
import os
OUTPUT_PATH = "/content/drive/MyDrive/predictive-analysis-data/regression/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save the scaled features with its metadata
train_final = pd.concat([train_meta.reset_index(drop=True),
                         X_train_scaled_df.reset_index(drop=True)], axis=1)
val_final = pd.concat([val_meta.reset_index(drop=True),
                       X_val_scaled_df.reset_index(drop=True)], axis=1)
test_final = pd.concat([test_meta.reset_index(drop=True),
                        X_test_scaled_df.reset_index(drop=True)], axis=1)

# export as csv
train_final.to_csv(f'{OUTPUT_PATH}train_regression.csv', index=False)
val_final.to_csv(f'{OUTPUT_PATH}val_regression.csv', index=False)
test_final.to_csv(f'{OUTPUT_PATH}test_regression.csv', index=False)


In [ ]:
# saving feature names metadata
feature_metadata = {
    'feature_columns': feature_cols,
    'n_features': len(feature_cols),
    'rolling_features': rolling_features,
    'lag_features': lag_features,
    'event_features': event_features,
    'machine_features': machine_features,
    'target_column': 'RUL_hours',
    'target_type': 'regression'
}

with open(f'{OUTPUT_PATH}regression_feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"Saved feature metadata")


Saved feature metadata


In [ ]:
# saving split metadata
split_metadata = {
    'dataset_info': {
        'total_rows': len(df),
        'date_range': {
            'start': str(df['datetime'].min()),
            'end': str(df['datetime'].max())
        }
    },
    'split_info': {
        'train': {
            'rows': len(train_df),
            'date_range': f"{train_df['datetime'].min()} to {train_df['datetime'].max()}",
            'rul_mean': float(train_df['RUL_hours'].mean()),
            'rul_median': float(train_df['RUL_hours'].median()),
            'rul_std': float(train_df['RUL_hours'].std())
        },
        'val': {
            'rows': len(val_df),
            'date_range': f"{val_df['datetime'].min()} to {val_df['datetime'].max()}",
            'rul_mean': float(val_df['RUL_hours'].mean()),
            'rul_median': float(val_df['RUL_hours'].median()),
            'rul_std': float(val_df['RUL_hours'].std())
        },
        'test': {
            'rows': len(test_df),
            'date_range': f"{test_df['datetime'].min()} to {test_df['datetime'].max()}",
            'rul_mean': float(test_df['RUL_hours'].mean()),
            'rul_median': float(test_df['RUL_hours'].median()),
            'rul_std': float(test_df['RUL_hours'].std())
        }
    },
    'scaling_method': 'StandardScaler',
    'target_type': 'regression'
}

with open(f'{OUTPUT_PATH}regression_split_metadata.json', 'w') as f:
    json.dump(split_metadata, f, indent=2)

print(f"Saved split metadata")


Saved split metadata
